# ☕ Coffee order parser: training notebook

Fine-tunes Qwen3.5-0.8B (Unsloth + LoRA, 1 epoch) on a synthetic dataset of spoken
coffee orders, and uploads a Q8_0 GGUF to Hugging Face for Ollama.

Design notes:
- **Compact output**: `cap L oat;lat x2` — about 3× fewer tokens than JSON, so faster
- **Refusals**: non-orders and off-menu items → `?`
- **Natural phrasing**: diminutives, digits, noise words ("с собой"), size corrections
- **1 epoch** to reduce memorization

Runtime → Change runtime type → **T4 GPU**, then Run all. Set `HF_USER` in step 8 first.

## 1. Install

In [ ]:
!pip install --upgrade -q unsloth unsloth_zoo
!pip install --upgrade -q "transformers>=5.0.0"
# If this cell errors, copy the install cell from Unsloth's official Qwen3.5 0.8B notebook.

## 2. Synthetic dataset

Output format: items separated by `;`, each `drink [size] [milk] [syrup] [xN]`. `?` = unclear / not on menu.

The generator lives in `src/coffee_parser/generator.py`. `production_dataset()` reproduces exactly the 4000 examples this model was trained on;
`tests/test_generator.py` pins that against a frozen copy of the original inline code.

In [ ]:
import sys
sys.path.insert(0, "/content/coffee-llm/src")   # path to the repo's src/

from coffee_parser import generator as gen

gen.assert_menu_consistent()
train_data, val_data = gen.production_dataset()   # same 4000 examples, seed 42
print(len(train_data), "train /", len(val_data), "val")
for ex in train_data[:12]:
    print(ex["input"], "\n  →", ex["output"])

## 3. Load model + LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bf16_supported

max_seq_length = 256

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3.5-0.8B",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4. Format (must match the TEMPLATE in `ollama/Modelfile` exactly)

In [ ]:
from datasets import Dataset

def prompt(inp):
    return f"<|im_start|>user\n{inp}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

def fmt(ex):
    return {"text": prompt(ex["input"]) + ex["output"] + "<|im_end|>\n"}

train_ds = Dataset.from_list(train_data).map(fmt)
print(train_ds[0]["text"])

## 5. Train (1 epoch)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        per_device_train_batch_size = 16,
        gradient_accumulation_steps = 1,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        warmup_steps = 10,
        lr_scheduler_type = "linear",
        logging_steps = 10,
        optim = "adamw_8bit",
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        output_dir = "outputs",
        seed = 3407,
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)
trainer.train()

## 6. Quick sanity check (the real test is `scripts/coffee_eval.py` on your Mac)

In [ ]:
FastLanguageModel.for_inference(model)

def parse(text):
    inputs = tokenizer(text=[prompt(text)], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

ok = sum(parse(ex["input"]) == ex["output"] for ex in val_data[:100])
print(f"Synthetic val: {ok}/100 (same generator, so expect it to be high)\n")

for t in ["добрый день можно капучинку побольше", "где у вас туалет", "капучино и круассан",
          "2 американо и 1 эспрессо", "can i get a large oat latte"]:
    print(f"{t:45} → {parse(t)}")

## 7. Export GGUF (Q8_0)

In [ ]:
import glob, os
model.save_pretrained_gguf("coffee", tokenizer, quantization_method = "q8_0")
gguf = [f for f in glob.glob("/content/**/*.gguf", recursive=True)
        if "Q8_0" in f and "mmproj" not in f and "coffee" in f]
print(gguf)
GGUF_PATH = max(gguf, key=os.path.getmtime)
print("Using:", GGUF_PATH)

## 8. Upload the GGUF to Hugging Face

In [ ]:
from huggingface_hub import HfApi, login
login()

HF_USER = "umarsheikh303"   # <- your username
api = HfApi()

gguf_repo = f"{HF_USER}/coffee-parser-gguf"
api.create_repo(gguf_repo, private=True, exist_ok=True)
api.upload_file(path_or_fileobj=GGUF_PATH, path_in_repo="coffee-q8_0.gguf", repo_id=gguf_repo)
print("Done:", gguf_repo)

## 9. On your Mac

```bash
cd ~/Development/Archive/coffee-llm

huggingface-cli download umarsheikh303/coffee-parser-gguf coffee-q8_0.gguf --local-dir models
ollama create coffee -f ollama/Modelfile

uv run python scripts/coffee_eval.py
```